In [14]:
# pick the existing design, self-generated semantic matrix
import numpy as np
import pandas as pd
import pickle

### Load Real Design

In [15]:
df = pd.read_csv("simu2b_data/exp1.csv")
df

,cycle,trial,phase,type,word1,word2,response,RT,correct,lag,serPos1,serPos2,subj,intactLag,prevResponse,prevRT
0,0,-1,study,intact,formal,positive,-1,-1.000,-1,-1,0,0,101,0,0,0
1,0,0,study,intact,skin,careful,-1,-1.000,-1,-1,1,1,101,0,0,0
2,0,1,study,intact,upon,miss,-1,-1.000,-1,-1,2,2,101,0,0,0
3,0,2,study,intact,single,tradition,-1,-1.000,-1,-1,3,3,101,0,0,0
4,0,3,study,intact,prove,airport,-1,-1.000,-1,-1,4,4,101,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107438,7,54,test,intact,typical,month,1,0.937,1,-1,10,10,213,2,1,0
107439,7,55,test,rearranged,miles,pretty,1,1.077,0,3,7,4,213,0,0,0
107440,7,56,test,rearranged,live,placed,1,0.633,0,5,52,47,213,0,0,0
107441,7,57,test,rearranged,cancer,matter,0,0.792,1,1,51,50,213,0,0,0


In [16]:
# add itemno
with open("simu2b_data/item2no.pkl", "rb") as f:
    item2no = pickle.load(f)
df["itemno1"] = df.apply(lambda x: item2no[x["word1"].lower().strip()], axis=1)
df["itemno2"] = df.apply(lambda x: item2no[x["word2"].lower().strip()], axis=1)
df

,cycle,trial,phase,type,word1,word2,response,RT,correct,lag,serPos1,serPos2,subj,intactLag,prevResponse,prevRT,itemno1,itemno2
0,0,-1,study,intact,formal,positive,-1,-1.000,-1,-1,0,0,101,0,0,0,422,759
1,0,0,study,intact,skin,careful,-1,-1.000,-1,-1,1,1,101,0,0,0,930,158
2,0,1,study,intact,upon,miss,-1,-1.000,-1,-1,2,2,101,0,0,0,1086,643
3,0,2,study,intact,single,tradition,-1,-1.000,-1,-1,3,3,101,0,0,0,923,1059
4,0,3,study,intact,prove,airport,-1,-1.000,-1,-1,4,4,101,0,0,0,793,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107438,7,54,test,intact,typical,month,1,0.937,1,-1,10,10,213,2,1,0,1079,650
107439,7,55,test,rearranged,miles,pretty,1,1.077,0,3,7,4,213,0,0,0,634,774
107440,7,56,test,rearranged,live,placed,1,0.633,0,5,52,47,213,0,0,0,586,738
107441,7,57,test,rearranged,cancer,matter,0,0.792,1,1,51,50,213,0,0,0,149,615


In [17]:
# discard subjects with less than 960 trials
df_count = df.groupby("subj").trial.count().to_frame().reset_index()
discard_subjs = df_count.query("trial != 960").subj.to_numpy()
discard_subjs

array([138])

In [18]:
df = df.query("subj not in @discard_subjs").copy()
df = df.drop(columns=["response", "RT", "correct", "serPos1", "serPos2", "intactLag", "prevResponse", "prevRT"])
df.rename(columns={"cycle": "list"}, inplace=True)
df

,list,trial,phase,type,word1,word2,lag,subj,itemno1,itemno2
0,0,-1,study,intact,formal,positive,-1,101,422,759
1,0,0,study,intact,skin,careful,-1,101,930,158
2,0,1,study,intact,upon,miss,-1,101,1086,643
3,0,2,study,intact,single,tradition,-1,101,923,1059
4,0,3,study,intact,prove,airport,-1,101,793,35
...,...,...,...,...,...,...,...,...,...,...
107438,7,54,test,intact,typical,month,-1,213,1079,650
107439,7,55,test,rearranged,miles,pretty,3,213,634,774
107440,7,56,test,rearranged,live,placed,5,213,586,738
107441,7,57,test,rearranged,cancer,matter,1,213,149,615


### Design

In [19]:
sess = np.unique(df.subj)
sess

array([ 101,  102,  103,  104,  105,  106,  107,  108,  109,  110,  111,
        112,  113,  114,  115,  116,  117,  118,  119,  120,  121,  122,
        123,  124,  125,  126,  127,  128,  129,  130,  131,  132,  133,
        134,  135,  136,  137,  139,  140,  141,  142,  143,  144,  145,
        146,  147,  148,  149,  151,  152,  153,  154,  155,  156,  157,
        158,  159,  160,  161,  162,  163,  164,  165,  166,  167,  168,
        169,  170,  171,  172,  173,  174,  175,  176,  177,  178,  179,
        180,  181,  182,  183,  184,  185,  186,  187,  188,  189,  190,
        192,  193,  194,  195,  196,  197,  198,  199,  200,  201,  202,
        203,  204,  205,  206,  207,  208,  209,  210,  211,  212,  213,
       1150])

In [20]:
df_study_lst = []
df_test_lst = []

for i in range(len(sess)):
    tmp = df.loc[df.subj == sess[i]].copy()
    tmp["session"] = i

    # study
    tmp_study = tmp.query("phase == 'study'").copy()
    tmp_study = tmp_study.drop(columns=["phase", "type", "lag"])
    df_study_lst.append(tmp_study)

    # test
    tmp_test = tmp.query("phase == 'test'").copy()
    tmp_test = tmp_test.drop(columns=["phase"])
    df_test_lst.append(tmp_test)

df_study = pd.concat(df_study_lst, ignore_index=True)
df_test = pd.concat(df_test_lst, ignore_index=True)
df_study.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)

In [21]:
df_study = df_study[["session", "subj", "list", "trial", "word1", "word2", "itemno1", "itemno2"]]
df_study

,session,subj,list,trial,word1,word2,itemno1,itemno2
0,0,101,0,-1,formal,positive,422,759
1,0,101,0,0,skin,careful,930,158
2,0,101,0,1,upon,miss,1086,643
3,0,101,0,2,single,tradition,923,1059
4,0,101,0,3,prove,airport,793,35
...,...,...,...,...,...,...,...,...
53275,110,1150,7,54,forget,pocket,419,752
53276,110,1150,7,55,patient,plastic,723,742
53277,110,1150,7,56,justice,lord,544,593
53278,110,1150,7,57,society,caught,940,165


In [22]:
df_test = df_test[["session", "subj", "list", "trial", "word1", "word2", "itemno1", "itemno2", "type", "lag"]]
df_test

,session,subj,list,trial,word1,word2,itemno1,itemno2,type,lag
0,0,101,0,-1,waste,degree,1112,269,rearranged,2
1,0,101,0,0,needed,able,673,2,rearranged,1
2,0,101,0,1,single,clean,923,193,rearranged,3
3,0,101,0,2,train,useful,1061,1089,rearranged,2
4,0,101,0,3,knees,various,552,1095,rearranged,5
...,...,...,...,...,...,...,...,...,...,...
53275,110,1150,7,54,opposite,period,703,729,intact,-1
53276,110,1150,7,55,card,limited,155,582,rearranged,2
53277,110,1150,7,56,soft,measure,941,620,rearranged,3
53278,110,1150,7,57,dark,fall,256,373,intact,-1


In [25]:
with open("simu2b_data/simu2b_design.pkl", "wb") as outp:
    pickle.dump(df_study, outp, pickle.HIGHEST_PROTOCOL)
    pickle.dump(df_test, outp, pickle.HIGHEST_PROTOCOL)